Install dependencies from uv.

In [1]:
!uv sync

Resolved 103 packages in 4ms
Checked 100 packages in 14ms


Load environment variables from .env file.

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

Create the weather tool.

In [3]:
import requests

def get_weather(latitude: float, longitude: float) -> dict:
    """
    Fetch weather forecast from the National Weather Service API.
    
    Args:
        latitude: The latitude of the location
        longitude: The longitude of the location
        
    Returns:
        A dictionary containing the weather forecast data
        
    Raises:
        requests.HTTPError: If the API request fails
        ValueError: If the coordinates are invalid
    """
    if not (-90 <= latitude <= 90) or not (-180 <= longitude <= 180):
        raise ValueError("Invalid coordinates. Latitude must be between -90 and 90, "
                         "longitude must be between -180 and 180.")

    headers = {
        "User-Agent": "WeatherApp/1.0 (your@email.com)",
        "Accept": "application/geo+json"
    }

    # Step 1: Get the grid points for the given coordinates
    points_url = f"https://api.weather.gov/points/{latitude},{longitude}"
    response = requests.get(points_url, headers=headers)
    response.raise_for_status()
    points_data = response.json()

    properties = points_data.get("properties", {})
    forecast_url = properties.get("forecast")
    location_info = {
        "city": properties.get("relativeLocation", {}).get("properties", {}).get("city"),
        "state": properties.get("relativeLocation", {}).get("properties", {}).get("state"),
        "grid_office": properties.get("gridId"),
    }

    if not forecast_url:
        raise ValueError("Could not retrieve forecast URL from NWS API.")

    # Step 2: Get the forecast using the forecast URL
    forecast_response = requests.get(forecast_url, headers=headers)
    forecast_response.raise_for_status()
    forecast_data = forecast_response.json()

    periods = forecast_data.get("properties", {}).get("periods", [])

    return {
        "location": location_info,
        "forecast": [
            {
                "name": period.get("name"),
                "temperature": period.get("temperature"),
                "temperature_unit": period.get("temperatureUnit"),
                "wind_speed": period.get("windSpeed"),
                "wind_direction": period.get("windDirection"),
                "short_forecast": period.get("shortForecast"),
                "detailed_forecast": period.get("detailedForecast"),
                "is_daytime": period.get("isDaytime"),
            }
            for period in periods
        ],
    }

Get coordinates for New York City.

In [4]:
get_weather(40.7143, -74.006)

{'location': {'city': 'New York', 'state': 'NY', 'grid_office': 'OKX'},
 'forecast': [{'name': 'This Afternoon',
   'temperature': 87,
   'temperature_unit': 'F',
   'wind_speed': '12 mph',
   'wind_direction': 'SW',
   'short_forecast': 'Slight Chance Showers And Thunderstorms',
   'detailed_forecast': 'A slight chance of showers and thunderstorms before 4pm. Mostly sunny. High near 87, with temperatures falling to around 85 in the afternoon. Heat index values as high as 97. Southwest wind around 12 mph. Chance of precipitation is 20%. New rainfall amounts less than a tenth of an inch possible.',
   'is_daytime': True},
  {'name': 'Tonight',
   'temperature': 77,
   'temperature_unit': 'F',
   'wind_speed': '6 to 10 mph',
   'wind_direction': 'SW',
   'short_forecast': 'Slight Chance Showers And Thunderstorms',
   'detailed_forecast': 'A slight chance of showers and thunderstorms before 2am. Partly cloudy, with a low around 77. Southwest wind 6 to 10 mph. Chance of precipitation is 20

Create tool for getting lattidue and longitude from google.

In [5]:
import urllib.request
import json
import os

GOOGLE_MAPS_KEY = os.getenv("GOOGLE_MAPS_KEY", "")

if not GOOGLE_MAPS_KEY:
    raise Exception("Google maps key must be set in environment.")

def get_lat_lon(city: str, state: str) -> tuple[float, float]:
    """
    Fetch latitude and longitude for a given city and state using Google Geocoding API.

    Args:
        city: City name (e.g. "Austin")
        state: State name or abbreviation (e.g. "TX" or "Texas")

    Returns:
        A tuple of (latitude, longitude)

    Raises:
        ValueError: If the location is not found or the API returns an error
    """
    address = urllib.parse.quote(f"{city}, {state}")
    url = f"https://maps.googleapis.com/maps/api/geocode/json?address={address}&key={GOOGLE_MAPS_KEY}"

    with urllib.request.urlopen(url) as response:
        data = json.loads(response.read().decode())

    if data["status"] != "OK":
        raise ValueError(f"Geocoding API error: {data['status']} for '{city}, {state}'")

    location = data["results"][0]["geometry"]["location"]
    return location["lat"], location["lng"]

Test getting lattidue and longitude.

In [6]:
get_lat_lon("New York City", "New York")

(40.7127753, -74.0059728)

Create the agent.

In [7]:
from google.adk.agents import Agent

weather_agent_instructions = """
You are an assistent to help with getting the weather. You will start by asking the user what city and state they want the weather for.
Use this to call the get_lat_long tool and get the latitude and longitude. Use this to call the get_weather tool to get the weather response.
Convert this response into human readable text.
"""

weather_agent = Agent(
    name="weather_agent",
    model="gemini-flash-latest",
    instruction=weather_agent_instructions,
    tools=[get_weather, get_lat_lon]
)

Setup the runner.

In [8]:
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types

session_service = InMemorySessionService()

runner = Runner(
    agent=weather_agent,
    app_name="weather_app",
    session_service=session_service,
)


async def run_prompt(prompt: str):
    session = await session_service.create_session(
        app_name="weather_app",
        user_id="user_123",
    )

    content = types.Content(
        role="user",
        parts=[types.Part(text=prompt)])

    async for event in runner.run_async(user_id="user_123",
                                        session_id=session.id,
                                        new_message=content):
        if event.is_final_response():
            print(event.content)

Perform some tests.

In [9]:
print("================ New York ===========================")
await run_prompt("What is the weather for New York City, New York")
print("================ Reston =============================")
await run_prompt("What is the weather for Reston, VA")
print("================ Los Angeles ========================")
await run_prompt("What is the weather for Los Angeles, CA")

================ New York ===========================


/Users/carpenterju/work/google-training/adk-workshop-jc/.venv/lib/python3.13/site-packages/google/adk/models/llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


parts=[Part(
  text="""Here is the current weather forecast for **New York City, NY**:

* **This Afternoon:** Mostly sunny with a slight chance of showers and thunderstorms before 4 PM. High near 87°F (heat index up to 97°F). Southwest winds around 12 mph. Chance of precipitation is 20%.
* **Tonight:** Partly cloudy with a slight chance of showers and thunderstorms before 2 AM. Low around 77°F. Southwest wind 6 to 10 mph.
* **Friday:** Mostly sunny becoming a chance of showers and thunderstorms after 2 PM. High near 91°F (heat index up to 103°F). Southwest wind 5 to 9 mph. Chance of precipitation is 40%.
* **Friday Night:** Mostly cloudy with a chance of showers and thunderstorms before 2 AM. Low around 76°F. Southwest wind around 7 mph. Chance of precipitation is 50%.
* **Saturday:** Mostly sunny with a high near 91°F and a slight chance of showers/thunderstorms after 2 PM. 
* **Sunday:** Sunny with a high near 91°F. Low around 75°F at night.""",
  thought_signature=b'\x12\x8e\x01\n\x